- ### fstream

1. [Read files](#read-files)

2. [Write files](#write-files)

3. [Read & write files](#read-files)

---

- ### Read files

In [1]:
# Setup for oneline command %%cpp
import os, tempfile, subprocess
from IPython.core.magic import register_cell_magic # type: ignore
import shlex

@register_cell_magic
def cpp(line, cell):
    """
    Usage:
    %%cpp -i "input for cin" -- arg1 arg2 ...
    """
    tokens = shlex.split(line)
    input_data = None
    run_args = []

    # Parse stdin input
    if "-i" in tokens:
        idx = tokens.index("-i")
        if idx + 1 < len(tokens):
            input_data = tokens[idx + 1]

    # Parse program arguments after --
    if "--" in tokens:
        idx = tokens.index("--")
        run_args = tokens[idx + 1:]

    # Write temp C++ file
    with tempfile.NamedTemporaryFile(suffix=".cpp", delete=False, mode="w") as tmp_cpp:
        tmp_cpp.write(cell)
        cpp_path = tmp_cpp.name
    exe_path = cpp_path[:-4] + ".exe"

    try:
        # Compile
        compile_proc = subprocess.run(
            ["g++", "-std=c++23", "-O2", "-Wall", cpp_path, "-o", exe_path],
            capture_output=True,
            text=True
        )
        if compile_proc.returncode != 0:
            print("❌ Compilation failed:\n", compile_proc.stderr)
            return

        # Run program
        run_proc = subprocess.run(
            [exe_path] + run_args,
            input=input_data,      # feed stdin here
            capture_output=True,
            text=True
        )
        if run_proc.stdout:
            print(run_proc.stdout, end="")
        if run_proc.stderr:
            print("⚠️ Runtime error:\n", run_proc.stderr)

    finally:
        for f in (cpp_path, exe_path):
            try: os.remove(f)
            except: pass

**ifstream**

In [6]:
%%cpp
#include <iostream>
#include <fstream>
#include <string>
using namespace std;

int main() {
    ifstream fin("fstream/input.txt");
    if (fin.is_open()) { // Check if file opened successfully
        string line;
        while (getline(fin, line)) { // Read line by line
            cout << line << endl;
        }
        fin.close();
    } else {
        cout << "Unable to open file";
    }
}

Hello, world!
Welcome!


---

- ### Write files

In [8]:
%%cpp
#include <iostream>
#include <fstream>
#include <string>
#include <vector>
using namespace std;

int main() {
    vector<string> lines = {
        "First line of text.",
        "Second line of text.",
        "Third line of text."
    };
    ofstream fout("fstream/output.txt");
    if (fout.is_open()) { // Check if file opened successfully
        for (const string& line : lines) {
            fout << line << endl; // Write each line to the file
        }
        fout.close();
    } else {
        cout << "Unable to write file";
    }
}

---

- ### Read & write files

In [15]:
%%cpp
#include <iostream>
#include <fstream>
#include <string>
#include <vector>
using namespace std;

int main() {
    vector<string> lines = {
        "First line of text.",
        "Second line of text.",
        "Third line of text."
    };
    fstream file("fstream/test.txt", ios::in | ios::out | ios::app); // app for append mode
    if (file.is_open()) { // Check if file opened successfully
        for (const string& line : lines) {
            file << line << endl; // Write each line to the file
        }
        file.seekg(0, ios::beg); // Move read pointer to the beginning
        string line;
        while (getline(file, line)) { // Read line by line
            cout << line << endl;
        }
        file.close();
    } else {
        cout << "Unable to open file";
    }
}


First line of text.
Second line of text.
Third line of text.
